# Statistical Validation of Train/Test Splits and Bootstrap Confidence Intervals

**Problem statement.** Reporting "my model gets 94% accuracy on the test set" is only trustworthy if two things hold: (1) the test set is actually drawn from the same distribution as production/training data, and (2) the single accuracy number isn't a fluke of that particular sample. As an ML QA / quality engineer, you're often the one who has to catch violations of both assumptions *before* a model ships. This notebook builds two reusable statistical checks that show up constantly in real ML testing and MLOps pipelines:

1. **Feature-level drift detection between two splits** using two-sample hypothesis tests (Kolmogorov–Smirnov + Welch's t-test) per feature, with a multiple-testing correction (Bonferroni) applied across all features — because testing 30 features at α=0.05 each will produce false alarms by chance alone if you don't correct for it.
2. **Bootstrap confidence intervals for evaluation metrics** (accuracy, F1) instead of a single point estimate, so you can say "F1 is 0.94, 95% CI [0.90, 0.97]" rather than implying false precision.

We use the real-world **Breast Cancer Wisconsin (Diagnostic)** dataset (`sklearn.datasets.load_breast_cancer`) — 569 real patient samples, 30 real-valued features computed from digitized images of breast masses, binary malignant/benign target.

**What you should produce:**
- `test_feature_drift(train_df, test_df, alpha=0.05)` — returns a DataFrame with one row per feature: KS-test p-value, t-test p-value, and a `flagged` boolean column indicating whether the feature is significantly different between the two splits *after* Bonferroni correction.
- `bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=1000, ci=0.95, random_state=0)` — returns `(point_estimate, lower_bound, upper_bound)` via the percentile bootstrap.
- Apply both functions to **two different splits** of the same dataset: (a) an ordinary random `train_test_split`, and (b) an adversarial split created by sorting on a feature before splitting (simulating the kind of silent data-pipeline bug — e.g. rows accidentally sorted by date or by a correlated column before a "random" split — that this drift check exists to catch). Your drift function should flag little-to-nothing on split (a) and flag several features on split (b).
- A short markdown conclusion (2-4 sentences) stating which features drifted on the adversarial split and why that matters for trusting an evaluation metric computed on it.


## Starter cells

Fill in the `TODO` sections below. Don't look at the solution until you've given each one a real attempt.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

RNG_SEED = 0
np.random.seed(RNG_SEED)

data = load_breast_cancer(as_frame=True)
df = data.frame.rename(columns={"target": "diagnosis"})
feature_cols = list(data.feature_names)

print(df.shape)
df.head()

### Step 1 — Build two splits of the same data

- `split_random`: an ordinary random 70/30 split (this is what a correct pipeline produces).
- `split_adversarial`: sort the full dataframe by `"worst area"` (a real feature, strongly correlated with several others) *before* splitting 70/30. This mimics a data pipeline bug where rows land in train/test non-randomly (e.g. sorted by an ID or a date that correlates with the target).

In [ ]:
def make_random_split(df, test_size=0.3, random_state=RNG_SEED):
    """Return (train_df, test_df) using an ordinary random split."""
    # TODO: use train_test_split on df (stratify on 'diagnosis')
    raise NotImplementedError

def make_adversarial_split(df, sort_col="worst area", test_size=0.3):
    """Return (train_df, test_df) after sorting df by sort_col, then
    taking the first (1-test_size) fraction as train and the rest as test.
    """
    # TODO: sort_values(sort_col), then slice by position
    raise NotImplementedError

# train_r, test_r = make_random_split(df)
# train_a, test_a = make_adversarial_split(df)

### Step 2 — Feature drift detector

For each feature column, compare its distribution in `train_df` vs `test_df` using:
- the two-sample Kolmogorov–Smirnov test (`scipy.stats.ks_2samp`) — sensitive to any distributional difference (shape, location, spread).
- Welch's t-test (`scipy.stats.ttest_ind(..., equal_var=False)`) — sensitive to a difference in means.

Then apply a **Bonferroni correction**: with `k` features tested, a feature is only `flagged` if its KS p-value is below `alpha / k`.

In [ ]:
def test_feature_drift(train_df, test_df, feature_cols=feature_cols, alpha=0.05):
    """Return a DataFrame indexed by feature name with columns:
    'ks_pvalue', 't_pvalue', 'bonferroni_alpha', 'flagged' (bool).
    A feature is flagged iff ks_pvalue < bonferroni_alpha.
    """
    # TODO:
    #  1. k = len(feature_cols); bonferroni_alpha = alpha / k
    #  2. for each column, run ks_2samp and ttest_ind(equal_var=False)
    #     between train_df[col] and test_df[col]
    #  3. collect results into a DataFrame, add 'flagged' column
    raise NotImplementedError

# drift_random = test_feature_drift(train_r, test_r)
# drift_adversarial = test_feature_drift(train_a, test_a)
# drift_random['flagged'].sum(), drift_adversarial['flagged'].sum()

### Step 3 — Bootstrap confidence interval for a metric

Train a simple `LogisticRegression` (with `max_iter=5000`) on `train_df`, predict on `test_df`, then compute a percentile bootstrap CI for a given metric by resampling **(y_true, y_pred) pairs together** with replacement `n_boot` times.

In [ ]:
def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=1000, ci=0.95, random_state=RNG_SEED):
    """Return (point_estimate, lower, upper) for metric_fn(y_true, y_pred)
    using the percentile bootstrap over resampled (y_true, y_pred) pairs.
    """
    # TODO:
    #  1. point = metric_fn(y_true, y_pred)
    #  2. rng = np.random.default_rng(random_state)
    #  3. resample indices with replacement, n_boot times, recompute metric_fn
    #  4. lower/upper = percentiles of the bootstrap distribution at
    #     (1-ci)/2 and 1-(1-ci)/2
    raise NotImplementedError

def fit_and_predict(train_df, test_df, feature_cols=feature_cols):
    """Fit LogisticRegression on train_df, return (y_test, y_pred) for test_df."""
    # TODO
    raise NotImplementedError

# y_test_r, y_pred_r = fit_and_predict(train_r, test_r)
# acc_ci_r = bootstrap_metric_ci(y_test_r, y_pred_r, accuracy_score)
# f1_ci_r = bootstrap_metric_ci(y_test_r, y_pred_r, f1_score)

### Step 4 — Put it together

Run the full pipeline on both splits and compare:
- How many features get flagged as drifted on the random split vs. the adversarial split?
- How do the bootstrap CIs for accuracy/F1 compare between the two splits — does the adversarial split's metric look artificially inflated, deflated, or just noisier?

Write your 2-4 sentence conclusion in a markdown cell below your results.

In [ ]:
# TODO: run make_random_split / make_adversarial_split, test_feature_drift,
# fit_and_predict, and bootstrap_metric_ci for both splits, then print/compare
# a summary table.

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

RNG_SEED = 0

def make_random_split(df, test_size=0.3, random_state=RNG_SEED):
    return train_test_split(
        df, test_size=test_size, random_state=random_state,
        stratify=df["diagnosis"],
    )

def make_adversarial_split(df, sort_col="worst area", test_size=0.3):
    sorted_df = df.sort_values(sort_col).reset_index(drop=True)
    n_test = int(len(sorted_df) * test_size)
    n_train = len(sorted_df) - n_test
    return sorted_df.iloc[:n_train], sorted_df.iloc[n_train:]

def test_feature_drift(train_df, test_df, feature_cols, alpha=0.05):
    k = len(feature_cols)
    bonferroni_alpha = alpha / k
    rows = []
    for col in feature_cols:
        a, b = train_df[col].values, test_df[col].values
        ks_stat, ks_p = stats.ks_2samp(a, b)
        t_stat, t_p = stats.ttest_ind(a, b, equal_var=False)
        rows.append({
            "feature": col,
            "ks_pvalue": ks_p,
            "t_pvalue": t_p,
            "bonferroni_alpha": bonferroni_alpha,
            "flagged": ks_p < bonferroni_alpha,
        })
    return pd.DataFrame(rows).set_index("feature")

def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=1000, ci=0.95, random_state=RNG_SEED):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    point = metric_fn(y_true, y_pred)
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    boot_stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_stats[i] = metric_fn(y_true[idx], y_pred[idx])
    alpha = 1 - ci
    lower, upper = np.percentile(boot_stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lower, upper

def fit_and_predict(train_df, test_df, feature_cols):
    clf = LogisticRegression(max_iter=5000)
    clf.fit(train_df[feature_cols], train_df["diagnosis"])
    y_pred = clf.predict(test_df[feature_cols])
    return test_df["diagnosis"].values, y_pred

# --- run on both splits ---
train_r, test_r = make_random_split(df)
train_a, test_a = make_adversarial_split(df)

drift_random = test_feature_drift(train_r, test_r, feature_cols)
drift_adversarial = test_feature_drift(train_a, test_a, feature_cols)

y_test_r, y_pred_r = fit_and_predict(train_r, test_r, feature_cols)
y_test_a, y_pred_a = fit_and_predict(train_a, test_a, feature_cols)

acc_ci_r = bootstrap_metric_ci(y_test_r, y_pred_r, accuracy_score)
acc_ci_a = bootstrap_metric_ci(y_test_a, y_pred_a, accuracy_score)
f1_ci_r = bootstrap_metric_ci(y_test_r, y_pred_r, f1_score)
f1_ci_a = bootstrap_metric_ci(y_test_a, y_pred_a, f1_score)

print("Flagged features — random split:", drift_random["flagged"].sum(), "/", len(feature_cols))
print("Flagged features — adversarial split:", drift_adversarial["flagged"].sum(), "/", len(feature_cols))
print("Accuracy CI (random):", acc_ci_r)
print("Accuracy CI (adversarial):", acc_ci_a)
print("F1 CI (random):", f1_ci_r)
print("F1 CI (adversarial):", f1_ci_a)
```

**Explanation.** On the random split, the KS/t-test pair should flag close to zero features (roughly `alpha * k` false positives are expected *before* correction, and the Bonferroni correction drives that to ~0). On the adversarial split — built by sorting on `worst area` before slicing — every feature correlated with `worst area` (most of the "worst"/"mean" size and texture features) gets flagged, because train now systematically contains the smaller-value rows and test the larger-value rows. Since `worst area` is also strongly predictive of malignancy, the adversarial test set ends up with a different class balance and feature distribution than train, so the reported accuracy/F1 on it is not a trustworthy estimate of real-world performance — it may look better or worse than the true model quality depending on which region of feature space test landed in. This is exactly the class of bug (silent non-random splitting, e.g. from an unshuffled upstream data source) that a drift check like `test_feature_drift` is meant to catch before an evaluation number is trusted.

</details>
